# Task 3: Domain Generalization Launcher
Mount your Google Drive and navigate to the Task 3 directory.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Auto-configured based on your Google Drive layout
DRIVE_ROOT = '/content/drive/MyDrive/atml_assignment1'
TASK3_DIR = os.path.join(DRIVE_ROOT, 'Task 3')
DATA_BASE = os.path.join(DRIVE_ROOT, 'data')

# Auto-detect the exact PACS folder inside 'data'
candidates = []
for root, dirs, files in os.walk(DATA_BASE):
    if set(['photo', 'art_painting', 'cartoon', 'sketch']).issubset(set(dirs)):
        candidates.append(root)
real = [c for c in candidates if 'dct' not in c.lower()]
DATA_DIR = real[0] if real else (candidates[0] if candidates else DATA_BASE)

# Task 3 uses Task 2's checkpoints for ERM baseline
SPLITS_PATH = os.path.join(TASK3_DIR, 'shared', 'splits', 'pacs_sketch_seed6304.json')
# Assuming you used 'Task 2' (capitalized) for your final checkpoints
TASK2_CKPT_DIR = os.path.join(DRIVE_ROOT, 'Task 2', 'checkpoints')
TASK3_CKPT_DIR = os.path.join(TASK3_DIR, 'checkpoints')
TASK3_RESULTS_DIR = os.path.join(TASK3_DIR, 'results')

os.makedirs(TASK3_CKPT_DIR, exist_ok=True)
os.makedirs(TASK3_RESULTS_DIR, exist_ok=True)

os.chdir(TASK3_DIR)
print(f'Working directory set to: {os.getcwd()}')
print(f'Data directory auto-detected as: {DATA_DIR}')


### Step 1: Train the Core Models (DAN-DG & SAM)

In [ ]:
!python train.py \
  --method dan_dg \
  --data_root "{DATA_DIR}" \
  --splits_path "{SPLITS_PATH}" \
  --checkpoints_dir "{TASK3_CKPT_DIR}" \
  --results_dir "{TASK3_RESULTS_DIR}"


In [ ]:
!python train.py \
  --method sam \
  --data_root "{DATA_DIR}" \
  --splits_path "{SPLITS_PATH}" \
  --checkpoints_dir "{TASK3_CKPT_DIR}" \
  --results_dir "{TASK3_RESULTS_DIR}"


### Step 2: Controlled Study (Lambda Sweep for DAN-DG)

In [ ]:
# Lambda = 0.1
!python train.py --method dan_dg --lambda_dg 0.1 \
  --data_root "{DATA_DIR}" \
  --splits_path "{SPLITS_PATH}" \
  --checkpoints_dir "{TASK3_CKPT_DIR}" \
  --results_dir "{TASK3_RESULTS_DIR}"


In [ ]:
# Lambda = 1.0 (Baseline)
!python train.py --method dan_dg --lambda_dg 1.0 \
  --data_root "{DATA_DIR}" \
  --splits_path "{SPLITS_PATH}" \
  --checkpoints_dir "{TASK3_CKPT_DIR}" \
  --results_dir "{TASK3_RESULTS_DIR}"


In [ ]:
# Lambda = 10.0
!python train.py --method dan_dg --lambda_dg 10.0 \
  --data_root "{DATA_DIR}" \
  --splits_path "{SPLITS_PATH}" \
  --checkpoints_dir "{TASK3_CKPT_DIR}" \
  --results_dir "{TASK3_RESULTS_DIR}"


### Step 3: Final Sketch Evaluation

In [ ]:
!python evaluate_sketch.py \
  --data_root "{DATA_DIR}" \
  --splits_path "{SPLITS_PATH}" \
  --checkpoints_dir "{TASK3_CKPT_DIR}" \
  --task2_checkpoints_dir "{TASK2_CKPT_DIR}" \
  --results_dir "{TASK3_RESULTS_DIR}"
